# L6 - Datadog Metrics, Cost Analysis & Operational Monitoring

## What You'll Learn

- How to derive operational metrics (throughput, latency, error rate) from APM spans
- How to compute real LLM cost from `gen_ai` token usage on the trace spans
- How to emit span-derived numbers as Datadog custom metrics
- How to build Monitors and an operational Dashboard on those metrics

## Prerequisites

- **Notebook 1 completed** (`1_datadog_multi_agent_tracing.ipynb`)
- **Step 2 below generates its own sample traffic** - no need to have run nb1 recently.
- App key with `apm_read` (span search). Steps 5-6 also need `monitors_write` /
  `dashboards_write` to create via API; **without them the notebook prints import-ready
  JSON** so you can create the monitors/dashboard in the UI instead.


## License Details

Refer to LICENSE file in the main folder for license details.

## How This Notebook Gets Its Data

This notebook is **self-sufficient** - it generates its own traffic (Step 2) and reads the
APM **spans** your agents send (via the L3 sidecar and the Harness's OTLP export), deriving
everything from them:

- **Sample traffic** (Step 2) - a couple of Harness invocations so the cells below have fresh spans.
- **Operational metrics** (Step 3) - throughput, latency percentiles, and error rate,
  computed client-side from span timing and status.
- **Cost** (Step 4) - real token usage read from the `gen_ai.usage.*` attributes on the
  `ConverseStream` LLM spans, multiplied by model pricing.
- **Custom metrics + Monitors + Dashboard** (Steps 5-6) - the span-derived numbers are
  emitted as Datadog **custom metrics** (`anycompany.*`), then monitored and dashboarded.

> **No AWS Integration required.** This track does not depend on `aws.bedrock.*` /
> `aws.lambda.*` CloudWatch metrics. In production the agents would emit these custom
> metrics continuously; here the notebook submits them from the span data to demonstrate
> the monitoring workflow end-to-end.


## Step 1: Setup

In [ ]:
import os, time, json
from datetime import datetime, timedelta, timezone
from collections import defaultdict

from datadog_api_client import Configuration, ApiClient
from datadog_api_client.v1.api.monitors_api import MonitorsApi
from datadog_api_client.v1.api.dashboards_api import DashboardsApi
from datadog_api_client.v2.api.spans_api import SpansApi
from datadog_api_client.v2.model.spans_list_request import SpansListRequest
from datadog_api_client.v2.model.spans_list_request_data import SpansListRequestData
from datadog_api_client.v2.model.spans_list_request_attributes import SpansListRequestAttributes
from datadog_api_client.v2.model.spans_list_request_page import SpansListRequestPage
from datadog_api_client.v2.model.spans_sort import SpansSort

DD_API_KEY = os.environ.get('DD_API_KEY', '')
DD_APP_KEY = os.environ.get('DD_APP_KEY', '')
DD_SITE = os.environ.get('DD_SITE', 'datadoghq.com')

if not DD_API_KEY or not DD_APP_KEY:
    import boto3
    ssm = boto3.client('ssm', region_name=os.environ.get('AWS_DEFAULT_REGION', 'us-east-1'))
    pfx = '/anycompany/agentcore'
    try:
        DD_API_KEY = ssm.get_parameter(Name=f'{pfx}/dd_api_key', WithDecryption=True)['Parameter']['Value']
        DD_APP_KEY = ssm.get_parameter(Name=f'{pfx}/dd_app_key', WithDecryption=True)['Parameter']['Value']
    except Exception as e:
        print(f'Set DD_API_KEY and DD_APP_KEY env vars. SSM error: {e}')

dd_config = Configuration()
dd_config.api_key['apiKeyAuth'] = DD_API_KEY
dd_config.api_key['appKeyAuth'] = DD_APP_KEY
dd_config.server_variables['site'] = DD_SITE

# Time window for this notebook (last 1 hour).
WINDOW_MIN = 60

# Claude Sonnet pricing (USD per 1M tokens). Adjust to your model/region if needed.
INPUT_PRICE = 3.00
OUTPUT_PRICE = 15.00

# Workshop service scope (orchestrator + specialist agents), minus /ping health checks.
SERVICE_QUERY = 'service:anycompany-* -resource_name:"GET /ping"'

def fetch_spans(query, minutes=WINDOW_MIN, limit=1000):
    # Return span attribute dicts for a Datadog span search (uses apm_read).
    _now = datetime.now(timezone.utc)
    with ApiClient(dd_config) as client:
        body = SpansListRequest(data=SpansListRequestData(type='search_request',
            attributes=SpansListRequestAttributes(
                filter={'from': (_now - timedelta(minutes=minutes)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                        'to': _now.strftime('%Y-%m-%dT%H:%M:%SZ'), 'query': query},
                sort=SpansSort.TIMESTAMP_DESCENDING,
                page=SpansListRequestPage(limit=limit))))
        resp = SpansApi(client).list_spans(body=body)
    return [s.get('attributes', {}) for s in resp.get('data', [])]

print(f'Datadog site: {DD_SITE}')
print(f'Window: last {WINDOW_MIN} min')
print(f'Service scope: {SERVICE_QUERY}')


## Step 2: Generate Sample Traffic

Invoke the Harness a couple of times so the metrics and cost cells below have fresh spans
to work with. This makes the notebook self-contained - you don't need to have run nb1
recently. Each invocation runs the full orchestration, so this may take a minute or two.


In [ ]:
# Generate a few real invocations so the metrics/cost cells below have fresh spans.
import os, time, uuid, boto3
from botocore.config import Config

_region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
_ssm = boto3.client('ssm', region_name=_region)
_cognito = boto3.client('cognito-idp', region_name=_region)
_agentcore = boto3.client('bedrock-agentcore', region_name=_region, config=Config(read_timeout=300))

def _gs(n):
    return _ssm.get_parameter(Name=n, WithDecryption=True)['Parameter']['Value']

_cfg = {k: _gs(f'/anycompany/agentcore/{k}') for k in ['harness_arn', 'cognito_client_id', 'user_password']}
_cognito.initiate_auth(ClientId=_cfg['cognito_client_id'], AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': 'gold_customer', 'PASSWORD': _cfg['user_password']})
print('Authenticated as gold_customer\n')

PROMPTS = [
    'What is the status of order ORD-10001?',          # order-agent path
    'Which of my orders are eligible for a refund?',   # refund-agent path
]
for p in PROMPTS:
    sid = str(uuid.uuid4()).upper()
    print(f'>>> {p}')
    resp = _agentcore.invoke_harness(harnessArn=_cfg['harness_arn'], runtimeSessionId=sid, actorId='CUST-789',
        messages=[{'role': 'user', 'content': [{'text': f'[Authenticated customer: CUST-789 | Tier: gold]\n\n{p}'}]}])
    out = ''
    for ev in resp['stream']:
        if 'contentBlockDelta' in ev:
            out += ev['contentBlockDelta'].get('delta', {}).get('text', '')
    print((out[:200] + '...') if out else '(no text returned)', '\n')

print('Waiting 60s for spans to reach Datadog...')
time.sleep(60)
print('Done - run the metrics/cost cells below.')


## Step 3: Operational Metrics from Traces

Derive throughput, error rate, and latency percentiles per service directly from the APM
spans - no `query_metrics`, no AWS Integration.


In [ ]:
# Operational metrics derived from APM spans (no AWS Integration, no metrics-query scope).
spans = fetch_spans(SERVICE_QUERY)
print(f'Fetched {len(spans)} spans across the workshop services (last {WINDOW_MIN} min).\n')

def _ms(s):
    st, en = s.get('start_timestamp'), s.get('end_timestamp')
    try:
        return (en - st).total_seconds() * 1000 if (st and en) else 0.0
    except Exception:
        return 0.0

def _pct(vals, p):
    if not vals:
        return 0.0
    vals = sorted(vals)
    k = (len(vals) - 1) * (p / 100.0)
    lo = int(k); hi = min(lo + 1, len(vals) - 1)
    return vals[lo] + (vals[hi] - vals[lo]) * (k - lo)

# A "request" = the inbound HTTP entry span for a service (type == 'web'): POST /invocations
# on the orchestrator, POST / on each specialist. Its duration is the end-to-end request
# latency. Child spans (CreateEvent, ai_guard, ConverseStream) are NOT requests, so they are
# not counted as requests nor averaged into request latency.
by_svc = defaultdict(lambda: {'spans': 0, 'errors': 0, 'req_ms': []})
for s in spans:
    d = by_svc[s.get('service', '?')]
    d['spans'] += 1
    if s.get('status') == 'error':
        d['errors'] += 1
    if s.get('type') == 'web':
        d['req_ms'].append(_ms(s))

print(f'{"Service":<28} {"Spans":>6} {"Reqs":>5} {"Err":>4} {"p50ms":>8} {"p95ms":>8} {"p99ms":>8}')
print('-' * 78)
for svc in sorted(by_svc):
    d = by_svc[svc]
    print(f'{svc:<28} {d["spans"]:>6} {len(d["req_ms"]):>5} {d["errors"]:>4} '
          f'{_pct(d["req_ms"], 50):>8.0f} {_pct(d["req_ms"], 95):>8.0f} {_pct(d["req_ms"], 99):>8.0f}')

if not spans:
    print('\nNo spans in the window - run Step 2 (generate traffic) above, then re-run.')


## Step 4: Cost from Token Usage

Sum the real `gen_ai.usage.input_tokens` / `output_tokens` recorded on every
`ConverseStream` LLM span and multiply by model pricing.


In [ ]:
# Cost from ACTUAL token usage on LLM spans (custom.gen_ai.usage.*). We filter client-side
# on the presence of gen_ai.usage rather than a resource-name wildcard, so we don't depend on
# the exact span name. Reuse Step 3's spans; if none carry token usage in the window, widen.
def _token_spans(span_list):
    out = []
    for s in span_list:
        usage = (s.get('custom', {}).get('gen_ai', {}) or {}).get('usage', {}) or {}
        if usage.get('input_tokens') is not None or usage.get('output_tokens') is not None:
            out.append(s)
    return out

llm_spans = _token_spans(spans)
if not llm_spans:
    print(f'No gen_ai (LLM) spans in the last {WINDOW_MIN} min - widening lookback to 6h...')
    llm_spans = _token_spans(fetch_spans('service:anycompany-*', minutes=360, limit=1000))

input_tokens = output_tokens = 0
models = set()
for s in llm_spans:
    gen_ai = s.get('custom', {}).get('gen_ai', {}) or {}
    usage = gen_ai.get('usage', {}) or {}
    input_tokens += int(usage.get('input_tokens', 0) or 0)
    output_tokens += int(usage.get('output_tokens', 0) or 0)
    m = (gen_ai.get('request', {}) or {}).get('model')
    if m:
        models.add(m)

# Invocations = orchestrator inbound HTTP (web) requests from Step 3; fall back to distinct traces.
try:
    invocations = len(by_svc.get('anycompany-orchestrator', {}).get('req_ms', [])) or 0
except NameError:
    invocations = 0
invocations = invocations or len({s.get('trace_id') for s in llm_spans}) or 1

input_cost = (input_tokens / 1_000_000) * INPUT_PRICE
output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE
total_cost = input_cost + output_cost

print('Cost from token usage')
print('=' * 52)
print(f'  Model(s):       {", ".join(sorted(models)) or "n/a"}')
print(f'  LLM calls:      {len(llm_spans):>12,}')
print(f'  Invocations:    {invocations:>12,}')
print(f'  Input tokens:   {input_tokens:>12,}  (${input_cost:.4f})')
print(f'  Output tokens:  {output_tokens:>12,}  (${output_cost:.4f})')
print(f'  {"-" * 44}')
print(f'  Total cost:     ${total_cost:.4f}')
print(f'  Per invocation: ${total_cost / invocations:.4f}')
print(f'  Projected / 1,000 sessions: ${total_cost / invocations * 1000:.2f}')
if not llm_spans:
    print('\n  No LLM spans found even over 6h - run Step 2 (generate traffic) above first.')

COST_METRICS = {'input_tokens': input_tokens, 'output_tokens': output_tokens,
                'total_cost_usd': total_cost, 'invocations': invocations}


## Step 5: Custom Metrics + Monitors

Emit the span-derived numbers as Datadog **custom metrics** (`anycompany.*`), then create
Monitors on them. The submit uses the API key (no app-key scope needed). Run once -
monitors persist in your Datadog account.


In [ ]:
# --- 5a: emit the span-derived numbers as Datadog CUSTOM METRICS (anycompany.*) ---
# Submission uses the API key intake (no app-key scope needed). In production the agents
# would emit these continuously; here we submit from the span data to close the loop.
try:
    from datadog_api_client.v2.api.metrics_api import MetricsApi as MetricsApiV2
    from datadog_api_client.v2.model.metric_payload import MetricPayload
    from datadog_api_client.v2.model.metric_series import MetricSeries
    from datadog_api_client.v2.model.metric_point import MetricPoint
    from datadog_api_client.v2.model.metric_intake_type import MetricIntakeType

    ts = int(time.time())
    base_tags = ['env:workshop', 'team:agentcore']
    series = [
        MetricSeries(metric='anycompany.llm.input_tokens', type=MetricIntakeType.COUNT,
                     points=[MetricPoint(timestamp=ts, value=float(COST_METRICS['input_tokens']))], tags=base_tags),
        MetricSeries(metric='anycompany.llm.output_tokens', type=MetricIntakeType.COUNT,
                     points=[MetricPoint(timestamp=ts, value=float(COST_METRICS['output_tokens']))], tags=base_tags),
        MetricSeries(metric='anycompany.llm.cost_usd', type=MetricIntakeType.GAUGE,
                     points=[MetricPoint(timestamp=ts, value=float(COST_METRICS['total_cost_usd']))], tags=base_tags),
    ]
    # Per-service throughput, errors, and p95 latency (drive the by-service dashboard widgets).
    for svc in sorted(by_svc):
        d = by_svc[svc]
        svc_tags = base_tags + [f'service:{svc}']
        series.append(MetricSeries(metric='anycompany.agent.requests', type=MetricIntakeType.COUNT,
                      points=[MetricPoint(timestamp=ts, value=float(len(d['req_ms'])))], tags=svc_tags))
        series.append(MetricSeries(metric='anycompany.agent.errors', type=MetricIntakeType.COUNT,
                      points=[MetricPoint(timestamp=ts, value=float(d['errors']))], tags=svc_tags))
        p95 = _pct(d['req_ms'], 95)
        if p95:
            series.append(MetricSeries(metric='anycompany.agent.latency_ms', type=MetricIntakeType.GAUGE,
                          points=[MetricPoint(timestamp=ts, value=float(p95))], tags=svc_tags))
    with ApiClient(dd_config) as client:
        MetricsApiV2(client).submit_metrics(body=MetricPayload(series=series))
    print(f'Submitted {len(series)} custom metric points (anycompany.*).')
except Exception as e:
    print(f'Custom metric submit skipped/failed (monitors still attempted below): {e}')

# --- 5b: create Monitors on the custom metrics ---
MONITORS = [
    {'name': '[AgentCore] LLM cost > $1 in last hour', 'type': 'metric alert',
     'query': 'sum(last_1h):sum:anycompany.llm.cost_usd{env:workshop} > 1',
     'message': 'LLM spend exceeded $1 in the last hour.',
     'tags': ['team:agentcore', 'env:workshop'], 'priority': 3,
     'options': {'thresholds': {'critical': 1, 'warning': 0.5}, 'notify_no_data': False}},
    {'name': '[AgentCore] Token budget > 500K/hour', 'type': 'metric alert',
     'query': 'sum(last_1h):sum:anycompany.llm.input_tokens{*}.as_count() + sum:anycompany.llm.output_tokens{*}.as_count() > 500000',
     'message': 'Token usage is high (>500K in 1 hour).',
     'tags': ['team:agentcore', 'env:workshop'], 'priority': 3,
     'options': {'thresholds': {'critical': 500000, 'warning': 250000}, 'notify_no_data': False}},
    {'name': '[AgentCore] Agent p95 latency > 60s', 'type': 'metric alert',
     'query': 'avg(last_15m):avg:anycompany.agent.latency_ms{*} > 60000',
     'message': 'Agent end-to-end p95 latency is high (>60s).',
     'tags': ['team:agentcore', 'env:workshop'], 'priority': 2,
     'options': {'thresholds': {'critical': 60000, 'warning': 30000}, 'notify_no_data': False}},
]
forbidden = False
with ApiClient(dd_config) as client:
    monitors_api = MonitorsApi(client)
    for m in MONITORS:
        try:
            result = monitors_api.create_monitor(body=m)
            print(f'  CREATED: {m["name"]} (id: {result.get("id", "?")})')
        except Exception as e:
            if 'Duplicate' in str(e):
                print(f'  ALREADY EXISTS: {m["name"]}')
            elif 'Forbidden' in str(e) or '403' in str(e):
                forbidden = True
                print(f'  SKIPPED (no monitors_write scope): {m["name"]}')
            else:
                print(f'  FAILED: {m["name"]} - {e}')

if forbidden:
    print('\nYour app key lacks the `monitors_write` scope, so the monitors were not created via API.')
    print('Create them in the UI (Monitoring > New Monitor), or add the scope and re-run.')
    print('Import-ready monitor definitions:')
    for m in MONITORS:
        print(json.dumps(m))

print(f'\nView the monitors: https://app.{DD_SITE}/monitors/manage')
print('  To find them, filter the Monitors list by tag:  team:agentcore   (or  env:workshop )')
print('  e.g. type  tag:team:agentcore  in the search box; all three are tagged env:workshop too.')


### Monitors in Datadog UI

The monitors created above are visible in **Monitoring > Monitor List**.
They will alert when thresholds are breached.

![Datadog Monitors](datadog_monitor.png)


## Step 6: Create Operational Dashboard

Create a Datadog dashboard from the `anycompany.*` custom metrics - token usage, cost,
and agent latency.


In [ ]:
# Operational dashboard built on the anycompany.* custom metrics from Step 4.
DASHBOARD = {
    'title': 'AnyCompany AgentCore - Multi-Agent Observability',
    'description': 'Span-derived operational + cost metrics for the multi-agent customer support system.',
    'layout_type': 'ordered',
    'widgets': [
        {'definition': {'title': 'LLM Token Usage (input vs output)', 'type': 'timeseries',
            'requests': [
                {'q': 'sum:anycompany.llm.input_tokens{*}.as_count()', 'display_type': 'bars'},
                {'q': 'sum:anycompany.llm.output_tokens{*}.as_count()', 'display_type': 'bars'}]}},
        {'definition': {'title': 'LLM Cost (USD)', 'type': 'timeseries',
            'requests': [{'q': 'sum:anycompany.llm.cost_usd{*}', 'display_type': 'line'}]}},
        {'definition': {'title': 'Total LLM Cost (USD)', 'type': 'query_value',
            'requests': [{'q': 'sum:anycompany.llm.cost_usd{*}', 'aggregator': 'sum'}]}},
        {'definition': {'title': 'Requests by Service', 'type': 'timeseries',
            'requests': [{'q': 'sum:anycompany.agent.requests{*} by {service}.as_count()', 'display_type': 'bars'}]}},
        {'definition': {'title': 'Errors by Service', 'type': 'timeseries',
            'requests': [{'q': 'sum:anycompany.agent.errors{*} by {service}.as_count()', 'display_type': 'bars'}]}},
        {'definition': {'title': 'Agent p95 Latency by Service (ms)', 'type': 'timeseries',
            'requests': [{'q': 'avg:anycompany.agent.latency_ms{*} by {service}', 'display_type': 'line'}]}},
    ],
}

with ApiClient(dd_config) as client:
    dashboards_api = DashboardsApi(client)
    try:
        result = dashboards_api.create_dashboard(body=DASHBOARD)
        print(f'Dashboard created: https://app.{DD_SITE}{result.get("url", "")}')
    except Exception as e:
        if 'Forbidden' in str(e) or '403' in str(e):
            print('Your app key lacks the `dashboards_write` scope - dashboard not created via API.')
            print('Import it in the UI: Dashboards > New Dashboard > (gear) Import Dashboard JSON,')
            print('paste the JSON below, or add the scope and re-run.\n')
            print(json.dumps(DASHBOARD, indent=2))
        else:
            print(f'Failed to create dashboard: {e}')

print('\nWidgets:')
for w in DASHBOARD['widgets']:
    print(f'  - {w["definition"]["title"]}')


### Dashboard in Datadog UI

The dashboard created above is visible in **Dashboards > Dashboard List**.

![Datadog Dashboard](datadog_dashboard.png)


## Summary

| What We Did | How |
|---|---|
| Operational metrics | Derived from APM spans (SpansApi) - latency, throughput, errors |
| Cost per session | `gen_ai.usage.*` token counts x Claude Sonnet pricing |
| Custom metrics | Submitted `anycompany.*` from span data (Metrics submit API) |
| Monitors | Datadog Monitors API on the custom metrics |
| Dashboard | Datadog Dashboards API (custom-metric widgets) |

### Next

Proceed to **`3_datadog_agent_observability.ipynb`** for AI-specific observability
(prompts, completions, tool decisions, evaluations).


## Cleanup

Uncomment and run the cell below to delete the resources this notebook creates:
- 3 Datadog Monitors (LLM cost, token budget, agent p95 latency) - matched by tag `team:agentcore`
- 1 Datadog Dashboard (AnyCompany AgentCore - Multi-Agent Observability) - matched by title

> The custom metrics (`anycompany.*`) are not deleted - Datadog metrics cannot be removed via
> API; they simply stop receiving points and age out. Deleting monitors/dashboards needs
> `monitors_write` / `dashboards_write`; if your key lacks them, delete them in the UI instead.


In [ ]:
# === CLEANUP: Uncomment all lines below and run this cell ===

# import os
# from datadog_api_client import Configuration, ApiClient
# from datadog_api_client.v1.api.monitors_api import MonitorsApi
# from datadog_api_client.v1.api.dashboards_api import DashboardsApi

# DD_API_KEY = os.environ.get('DD_API_KEY', '')
# DD_APP_KEY = os.environ.get('DD_APP_KEY', '')
# DD_SITE = os.environ.get('DD_SITE', 'datadoghq.com')

# # Load from SSM if not in env
# if not DD_API_KEY:
#     import boto3
#     ssm = boto3.client('ssm', region_name='us-east-1')
#     DD_API_KEY = ssm.get_parameter(Name='/anycompany/agentcore/dd_api_key', WithDecryption=True)['Parameter']['Value']
#     DD_APP_KEY = ssm.get_parameter(Name='/anycompany/agentcore/dd_app_key', WithDecryption=True)['Parameter']['Value']
#     DD_SITE = ssm.get_parameter(Name='/anycompany/agentcore/dd_site')['Parameter']['Value']

# dd_config = Configuration()
# dd_config.api_key['apiKeyAuth'] = DD_API_KEY
# dd_config.api_key['appKeyAuth'] = DD_APP_KEY
# dd_config.server_variables['site'] = DD_SITE

# # 1. Delete all monitors tagged with team:agentcore
# print('Deleting Datadog monitors...')
# with ApiClient(dd_config) as client:
#     monitors_api = MonitorsApi(client)
#     monitors = monitors_api.list_monitors(monitor_tags='team:agentcore')
#     if not monitors:
#         print('  No monitors found with tag team:agentcore')
#     for m in monitors:
#         try:
#             monitors_api.delete_monitor(m.id)
#             print(f'  Deleted: {m.name} (id: {m.id})')
#         except Exception as e:
#             print(f'  Error deleting {m.name}: {e}')

# # 2. Delete the dashboard (matched by title; created in Step 6)
# print('\nDeleting Datadog dashboard...')
# with ApiClient(dd_config) as client:
#     dashboards_api = DashboardsApi(client)
#     # List dashboards and find ours by title
#     all_dashes = dashboards_api.list_dashboards()
#     for d in all_dashes.dashboards:
#         if 'AnyCompany AgentCore' in (d.title or ''):
#             try:
#                 dashboards_api.delete_dashboard(d.id)
#                 print(f'  Deleted: {d.title} (id: {d.id})')
#             except Exception as e:
#                 print(f'  Error deleting {d.title}: {e}')

# print('\nCleanup complete.')
